# Quality Primary Education Index (QPEI)
## Framework-aligned analysis notebook

**Purpose:** This notebook implements the established QPEI methodology while adding the planned empirical measurement analyses.

### Analytical logic

**Data audit → translation/codebook → demographics → item diagnostics → EFA → reliability → CFA where defensible → school-level aggregation → six QPEI dimensions → theory-informed QPEI → robustness → demographic/explanatory analysis → tables/figures → results registry**

### Important methodological rule

The QPEI remains a **multidimensional formative composite**. EFA/CFA are used to investigate and validate the empirical structure of respondent-derived measurement batteries and to map observed indicators to the theoretical framework. They are **not** used to redefine the QPEI as a single reflective latent variable.

The six theoretical dimensions and their weights remain:

- D1 Teacher Competence & Pedagogical Practice: **20%**
- D2 Curriculum Implementation & Assessment: **15%**
- D3 Learning Environment & Infrastructure: **15%**
- D4 Student Learning & Foundational Learning: **20%**
- D5 School Leadership & Community Support: **15%**
- D6 Equity & Inclusion: **15%**

Indicators are equally weighted **within** dimensions; dimensions receive the framework weights.


## Colab execution

Run **Runtime → Run all**. The notebook now:

- mounts Google Drive;
- uses the corrected `DATA_PATH`, `RESULTS_PATH`, and `OUTPUT_DIR`;
- creates `tables`, `figures`, and `logs` subfolders;
- explicitly saves analysis outputs as CSV files;
- creates/updates the project `results.json`.

This avoids the problem where a notebook appears to run but produces no persistent files.

In [ ]:
# 0. SETUP — GOOGLE COLAB READY
!pip -q install factor_analyzer pingouin semopy openpyxl statsmodels scikit-learn seaborn

from pathlib import Path
import os, json, warnings, math, re
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings("ignore")

from google.colab import drive
drive.mount("/content/drive")

DATA_PATH = Path("/content/drive/MyDrive/QPEI/QPE_MASTER_Cleaned_IDs_Translated_Analysis.xlsx")
RESULTS_PATH = Path("/content/drive/MyDrive/QPEI/results.json")
OUTPUT_DIR = Path("/content/drive/MyDrive/QPEI/qpei_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TABLE_DIR = OUTPUT_DIR / "tables"
FIGURE_DIR = OUTPUT_DIR / "figures"
LOG_DIR = OUTPUT_DIR / "logs"
for p in [TABLE_DIR, FIGURE_DIR, LOG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

from factor_analyzer import FactorAnalyzer
from factor_analyzer.factor_analyzer import calculate_kmo, calculate_bartlett_sphericity
import pingouin as pg
from semopy import Model, calc_stats

print("="*70)
print("QPEI ANALYSIS INITIALIZATION")
print("="*70)
print("DATA_PATH:", DATA_PATH)
print("Exists:", DATA_PATH.exists())
print("RESULTS_PATH:", RESULTS_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)

if not DATA_PATH.exists():
    raise FileNotFoundError(f"MASTER FILE NOT FOUND: {DATA_PATH}")

print("Setup complete.")


## 1. Load and audit the master workbook

The first rule is **never overwrite the original responses**. Cleaning creates canonical analysis variables while preserving source fields.

The cleaned school ID is authoritative for school-level aggregation.


In [ ]:
# 1. LOAD WORKBOOK
xl = pd.ExcelFile(DATA_PATH)
print("Workbook:", DATA_PATH)
print("Sheets:", xl.sheet_names)

sheets = {s: pd.read_excel(DATA_PATH, sheet_name=s) for s in xl.sheet_names}

inventory = pd.DataFrame([
    {
        "sheet": name,
        "rows": len(df),
        "columns": len(df.columns),
        "missing_cells": int(df.isna().sum().sum()),
        "duplicate_rows": int(df.duplicated().sum())
    }
    for name, df in sheets.items()
])

display(inventory)
inventory.to_csv(TABLE_DIR / "workbook_inventory.csv", index=False)
print("Saved workbook_inventory.csv")


In [ ]:
# 1A. BASIC WORKBOOK AUDIT
audit = inventory.copy()
display(audit)
audit.to_csv(TABLE_DIR / "Table_01_workbook_audit.csv", index=False)
print("Saved:", TABLE_DIR / "Table_01_workbook_audit.csv")


## 2. School-ID reconciliation

Use the established 32-school framework. **Do not infer a school from a malformed range ID alone.** Prefer the canonical school master / school name / enumerator mapping.

The corrected ID should be used for aggregation, while the original ID remains available for audit.


In [ ]:
# 2. BUILD A CANONICAL SCHOOL MASTER
# This section is intentionally explicit. Populate/verify against the School_Master
# produced during the cleaning stage.

school_master_candidates = []
for s in ["Teacher", "Parent", "Student", "Classroom_Observation", "School_Environment"]:
    if s in sheets:
        df = sheets[s]
        cols = [c for c in ["school_id","school_name","enumerator_id","source_enumerator"] if c in df.columns]
        if cols:
            tmp = df[cols].copy()
            tmp["source_sheet"] = s
            school_master_candidates.append(tmp)

if school_master_candidates:
    school_refs = pd.concat(school_master_candidates, ignore_index=True)
    display(school_refs.head())
else:
    print("Check exact sheet names/column names before proceeding.")


### School-ID correction policy

Known structural corrections from the master audit are encoded only where the school identity is supported by the school name/enumerator mapping:

- E03 → S09–S12
- E04 → S13–S16
- E05 → S17–S20
- E06 → S21–S24

Malformed range strings such as `S05-S08` or `S21-S24` are therefore not treated as literal school IDs.

**Do not silently apply a correction when the mapping is ambiguous. Flag it for review.**


In [ ]:
# 2A. Generic ID normalization
def normalize_school_id(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip().upper()
    # Known single-ID formatting
    import re
    m = re.fullmatch(r"S0*(\d+)", x)
    if m:
        return f"S{int(m.group(1)):02d}"
    return x

# Apply to any school_id column while preserving the original.
for name, df in sheets.items():
    if "school_id" in df.columns:
        df["school_id_original"] = df["school_id"]
        df["school_id_clean"] = df["school_id"].apply(normalize_school_id)

print("School-ID normalization complete.")


## 3. Translation and framework item map

The analysis must keep the original Bangla item wording intact. English translations should be stored in a separate dictionary, not substituted for the original responses.

**Do not invent item meanings from variable names.** Populate the item dictionary from the survey instrument/codebook.

The framework mapping should identify whether an item is:

- candidate QPEI indicator
- demographic/context variable
- open-ended qualitative item
- administrative identifier
- observation/environment indicator


In [ ]:
# 3. ITEM MAP TEMPLATE
item_map = pd.DataFrame(columns=[
    "source", "item_code", "original_text_bn", "translation_en",
    "response_scale", "candidate_factor", "qpei_dimension",
    "direction", "efa_eligible", "cfa_eligible", "notes"
])

# Load a codebook sheet if present and inspect it before filling the map.
for candidate in ["Codebook", "Questionnaire", "Survey_Codebook", "Item_Map"]:
    if candidate in sheets:
        print("Potential codebook:", candidate)
        display(sheets[candidate].head(20))

# IMPORTANT:
# Fill item_map from the actual questionnaire/codebook before naming factors.


## 4. Demographic analysis

Report descriptive characteristics separately for teachers, students, parents and schools.

Categorical variables: **n and %**.

Continuous variables: **mean, SD, median, IQR, range** where appropriate.

Do not include demographics in EFA.


In [ ]:
# 4. DEMOGRAPHIC SUMMARY HELPER
def categorical_summary(df, col):
    x = df[col].dropna()
    out = x.value_counts(dropna=False).rename_axis(col).reset_index(name="n")
    out["percent"] = out["n"] / len(x) * 100
    return out

def continuous_summary(df, col):
    x = pd.to_numeric(df[col], errors="coerce").dropna()
    return pd.Series({
        "n": len(x), "mean": x.mean(), "sd": x.std(ddof=1),
        "median": x.median(), "iqr": x.quantile(.75)-x.quantile(.25),
        "min": x.min(), "max": x.max()
    })

# Inspect columns first; select demographic variables from the real workbook.
for name in ["Teacher", "Student", "Parent"]:
    if name in sheets:
        print("\n", name)
        print(list(sheets[name].columns))


## 5. Item-level diagnostics

For each candidate measurement item:

- missing %
- mean
- SD
- median
- min/max
- skewness
- kurtosis
- floor effect
- ceiling effect

Flag extreme concentration, but do not automatically delete theoretically important formative indicators.


In [ ]:
# 5. ITEM DIAGNOSTICS
def item_diagnostics(df, item_cols):
    rows = []
    for c in item_cols:
        x = pd.to_numeric(df[c], errors="coerce").dropna()
        if len(x) == 0:
            continue
        rows.append({
            "item": c,
            "n": len(x),
            "missing_pct": df[c].isna().mean()*100,
            "mean": x.mean(),
            "sd": x.std(ddof=1),
            "median": x.median(),
            "min": x.min(),
            "max": x.max(),
            "skewness": stats.skew(x, bias=False) if len(x)>2 else np.nan,
            "kurtosis": stats.kurtosis(x, bias=False) if len(x)>3 else np.nan,
            "floor_pct": (x == x.min()).mean()*100,
            "ceiling_pct": (x == x.max()).mean()*100
        })
    return pd.DataFrame(rows)

# Populate these from the actual workbook after inspecting the item map.
TQ_ITEMS = [c for c in sheets.get("Teacher", pd.DataFrame()).columns if str(c).upper().startswith("TQ")]
SQ_ITEMS = [c for c in sheets.get("Student", pd.DataFrame()).columns if str(c).upper().startswith("SQ")]
PQ_ITEMS = [c for c in sheets.get("Parent", pd.DataFrame()).columns if str(c).upper().startswith("PQ")]

teacher_diag = item_diagnostics(sheets["Teacher"], TQ_ITEMS) if "Teacher" in sheets else pd.DataFrame()
student_diag = item_diagnostics(sheets["Student"], SQ_ITEMS) if "Student" in sheets else pd.DataFrame()
parent_diag = item_diagnostics(sheets["Parent"], PQ_ITEMS) if "Parent" in sheets else pd.DataFrame()

display(teacher_diag)
display(student_diag)
display(parent_diag)


# 6. Exploratory Factor Analysis

### Prespecified EFA method

- Principal Axis Factoring (PAF)
- Oblique rotation: Oblimin initially
- Factor number: Parallel Analysis + theoretical interpretability
- KMO and Bartlett's test before extraction
- Approximate loading screening threshold: |.40|
- Cross-loadings are flagged, not automatically deleted
- Communalities are reviewed alongside substantive importance

**Teacher, Student, and Parent batteries are analysed separately.**

Classroom Observation (n=8) and School Environment (n=32) are not subjected to conventional item-level EFA.


In [ ]:
# 6A. KMO + Bartlett
def factorability_report(df, items):
    X = df[items].apply(pd.to_numeric, errors="coerce").dropna()
    if X.shape[0] < 5 or X.shape[1] < 2:
        return None
    kmo_all, kmo_model = calculate_kmo(X)
    chi2, p = calculate_bartlett_sphericity(X)
    return {
        "n": X.shape[0],
        "items": X.shape[1],
        "KMO": kmo_model,
        "Bartlett_chi2": chi2,
        "Bartlett_p": p
    }

if FACTOR_ANALYZER_OK:
    for label, df, items in [
        ("Teacher", sheets.get("Teacher"), TQ_ITEMS),
        ("Student", sheets.get("Student"), SQ_ITEMS),
        ("Parent", sheets.get("Parent"), PQ_ITEMS)
    ]:
        if df is not None and items:
            print(label, factorability_report(df, items))


In [ ]:
# 6B. Parallel analysis
def parallel_analysis(df, items, n_iter=500, random_state=42):
    rng = np.random.default_rng(random_state)
    X = df[items].apply(pd.to_numeric, errors="coerce").dropna().to_numpy()
    X = (X - X.mean(axis=0)) / X.std(axis=0, ddof=1)
    corr = np.corrcoef(X, rowvar=False)
    obs = np.linalg.eigvalsh(corr)[::-1]

    simulated = np.zeros((n_iter, X.shape[1]))
    n, p = X.shape
    for i in range(n_iter):
        Z = rng.normal(size=(n, p))
        rc = np.corrcoef(Z, rowvar=False)
        simulated[i] = np.linalg.eigvalsh(rc)[::-1]

    mean_sim = simulated.mean(axis=0)
    p95_sim = np.percentile(simulated, 95, axis=0)
    suggested = int(np.sum(obs > mean_sim))

    return pd.DataFrame({
        "factor": np.arange(1, len(obs)+1),
        "observed_eigenvalue": obs,
        "mean_random_eigenvalue": mean_sim,
        "random_95pct": p95_sim
    }), suggested

parallel_results = {}
if FACTOR_ANALYZER_OK:
    for label, df, items in [
        ("Teacher", sheets.get("Teacher"), TQ_ITEMS),
        ("Student", sheets.get("Student"), SQ_ITEMS),
        ("Parent", sheets.get("Parent"), PQ_ITEMS)
    ]:
        if df is not None and len(items) >= 3:
            pa, k = parallel_analysis(df, items)
            parallel_results[label] = (pa, k)
            print(label, "suggested factors:", k)
            display(pa.head(10))


In [ ]:
# 6C. EFA extraction
def run_efa(df, items, n_factors):
    X = df[items].apply(pd.to_numeric, errors="coerce").dropna()
    fa = FactorAnalyzer(
        n_factors=n_factors,
        method="principal",
        rotation="oblimin"
    )
    fa.fit(X)

    loadings = pd.DataFrame(
        fa.loadings_,
        index=items,
        columns=[f"Factor_{i+1}" for i in range(n_factors)]
    )
    communalities = pd.Series(fa.get_communalities(), index=items, name="communality")
    uniqueness = pd.Series(fa.get_uniquenesses(), index=items, name="uniqueness")
    return fa, loadings, pd.concat([communalities, uniqueness], axis=1)

efa_results = {}
for label, df, items in [
    ("Teacher", sheets.get("Teacher"), TQ_ITEMS),
    ("Student", sheets.get("Student"), SQ_ITEMS),
    ("Parent", sheets.get("Parent"), PQ_ITEMS)
]:
    if label in parallel_results:
        n_factors = parallel_results[label][1]
        if n_factors >= 1:
            efa_results[label] = run_efa(df, items, n_factors)
            print("\n", label)
            display(efa_results[label][1].round(3))
            display(efa_results[label][2].round(3))


## 7. Reliability

Reliability is applied to **empirically coherent respondent scales**, not mechanically to the entire formative QPEI.

We report alpha and omega, plus item-total diagnostics.


In [ ]:
# 7. RELIABILITY HELPER
def cronbach_alpha(df):
    X = df.apply(pd.to_numeric, errors="coerce").dropna()
    k = X.shape[1]
    if k < 2:
        return np.nan
    item_vars = X.var(axis=0, ddof=1)
    total_var = X.sum(axis=1).var(ddof=1)
    return k/(k-1) * (1 - item_vars.sum()/total_var)

# Omega requires an explicit factor model and will be calculated after final factor retention.


# 8. Confirmatory Factor Analysis

CFA is performed **after EFA**, only for models that are theoretically interpretable and statistically estimable.

CFA tests the measurement structure of respondent-derived scales. It does not convert the six-dimensional formative QPEI into a reflective latent variable.

No modification-index-driven model fishing.


In [ ]:
# 8A. CFA scaffold using semopy
# The model strings must be generated from the final EFA solution and the item map.
# Example:
#
# teacher_model = '''
# Factor1 =~ TQ6 + TQ7 + TQ8
# Factor2 =~ TQ15 + TQ16 + TQ17
# '''
#
# Then:
#
# from semopy import Model, calc_stats
# model = Model(teacher_model)
# model.fit(teacher_df)
# stats = calc_stats(model)
#
# Do not write the final model until EFA + item content have been reviewed.
print("CFA scaffold ready; final models will be generated from retained EFA items.")


# 9. School-level aggregation

Only indicators eligible for school-level aggregation are aggregated.

Evidence should be based on:

- ICC(1)
- ICC(2)
- rWG where appropriate
- number of respondents per school
- substantive appropriateness

The observation and school-environment instruments are already closer to school/classroom level and should not be treated as ordinary individual respondent scales.


In [ ]:
# 9. ICC helpers
def one_way_icc(df, value_col, group_col):
    d = df[[value_col, group_col]].dropna()
    groups = d[group_col].unique()
    k = d.groupby(group_col).size().mean()
    grand = d[value_col].mean()
    group_stats = d.groupby(group_col)[value_col].agg(["mean","count"])

    between = sum(row["count"]*(row["mean"]-grand)**2 for _, row in group_stats.iterrows())
    dfb = len(groups)-1
    msb = between/dfb if dfb > 0 else np.nan

    within = 0
    for g, sub in d.groupby(group_col):
        within += ((sub[value_col]-sub[value_col].mean())**2).sum()
    dfw = len(d)-len(groups)
    msw = within/dfw if dfw > 0 else np.nan

    icc1 = (msb-msw)/(msb+(k-1)*msw) if (msb+(k-1)*msw) != 0 else np.nan
    icc2 = (msb-msw)/msb if msb != 0 else np.nan
    return icc1, icc2

print("Aggregation helpers defined.")


# 10. QPEI construction

### Framework-prespecified dimensions

| Code | Dimension | Weight |
|---|---|---:|
| D1 | Teacher Competence & Pedagogical Practice | 0.20 |
| D2 | Curriculum Implementation & Assessment | 0.15 |
| D3 | Learning Environment & Infrastructure | 0.15 |
| D4 | Student Learning & Foundational Learning | 0.20 |
| D5 | School Leadership & Community Support | 0.15 |
| D6 | Equity & Inclusion | 0.15 |

The factor analysis can inform **which indicators empirically cohere and how they map to these dimensions**, but the framework weights are not estimated from the sample.


In [ ]:
# 10A. QPEI framework object
QPEI_WEIGHTS = {
    "D1": 0.20,
    "D2": 0.15,
    "D3": 0.15,
    "D4": 0.20,
    "D5": 0.15,
    "D6": 0.15
}

QPEI_NAMES = {
    "D1": "Teacher Competence & Pedagogical Practice",
    "D2": "Curriculum Implementation & Assessment",
    "D3": "Learning Environment & Infrastructure",
    "D4": "Student Learning & Foundational Learning",
    "D5": "School Leadership & Community Support",
    "D6": "Equity & Inclusion"
}

assert abs(sum(QPEI_WEIGHTS.values()) - 1) < 1e-12


In [ ]:
# 10B. Normalize Likert indicators to 0-100
def likert_to_100(x, min_score=1, max_score=5, reverse=False):
    x = pd.to_numeric(x, errors="coerce")
    if reverse:
        x = max_score + min_score - x
    return (x-min_score)/(max_score-min_score)*100

def dimension_score(df, items, reverse_map=None):
    reverse_map = reverse_map or {}
    scored = pd.DataFrame({
        item: likert_to_100(df[item], reverse=reverse_map.get(item, False))
        for item in items
    })
    return scored.mean(axis=1, skipna=True)

print("Normalization functions ready.")


# 11. Robustness and sensitivity

Primary QPEI = theory-informed weights.

Sensitivity analyses:

1. Equal dimension weights
2. Entropy Weight Method (EWM)
3. CRITIC
4. TOPSIS
5. Monte Carlo perturbation of dimension weights
6. Complete-case versus coverage-adjusted QPEI

The purpose is to test whether conclusions are stable rather than to choose whichever weighting method produces the preferred ranking.


In [ ]:
# 11. Rank comparison helper
from scipy.stats import spearmanr, kendalltau

def rank_agreement(df_scores):
    methods = df_scores.columns
    rows = []
    for i, a in enumerate(methods):
        for b in methods[i+1:]:
            rho, rp = spearmanr(df_scores[a], df_scores[b], nan_policy="omit")
            tau, tp = kendalltau(df_scores[a], df_scores[b], nan_policy="omit")
            rows.append({
                "method_a": a, "method_b": b,
                "spearman_rho": rho, "spearman_p": rp,
                "kendall_tau": tau, "kendall_p": tp
            })
    return pd.DataFrame(rows)


# 12. Demographic and explanatory analysis

The inferential unit must match the data-generating level.

- Individual-level comparisons use respondent-level outcomes.
- QPEI regression uses school-level data.
- With **32 schools**, predictive models must remain parsimonious.

Report effect sizes and confidence intervals, not p-values alone.


In [ ]:
# 12. Generic group comparison helpers
def two_group_summary(df, outcome, group):
    d = df[[outcome, group]].dropna()
    levels = d[group].unique()
    if len(levels) != 2:
        raise ValueError("This helper requires exactly two groups.")
    a = d.loc[d[group] == levels[0], outcome]
    b = d.loc[d[group] == levels[1], outcome]
    t, p = stats.ttest_ind(a, b, equal_var=False, nan_policy="omit")
    return {
        "group_1": levels[0], "n1": len(a), "mean1": a.mean(),
        "group_2": levels[1], "n2": len(b), "mean2": b.mean(),
        "t": t, "p": p
    }


# 13. Tables and figures

All manuscript outputs should be generated from the same objects used for analysis.

Recommended key figures:

1. Analytical workflow
2. Sample composition
3. Item distributions/missingness
4–6. Parallel analysis plots
7. Factor-loading heatmap
8. Reliability profile
9–11. CFA diagrams
12. Aggregation evidence
13. QPEI architecture
14. QPEI distribution
15. Dimension profile
16. School ranking
17. Ranking robustness
18. Monte Carlo stability
19–20. Demographic/explanatory results
21. Integrated evidence model


In [ ]:
# 13A. Example parallel-analysis figure
def plot_parallel(pa, title):
    plt.figure(figsize=(8,5))
    plt.plot(pa["factor"], pa["observed_eigenvalue"], marker="o", label="Observed")
    plt.plot(pa["factor"], pa["mean_random_eigenvalue"], marker="o", label="Mean random")
    plt.plot(pa["factor"], pa["random_95pct"], linestyle="--", label="Random 95th percentile")
    plt.axhline(1, linestyle=":", linewidth=1)
    plt.xlabel("Factor")
    plt.ylabel("Eigenvalue")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()

# Example:
# plot_parallel(parallel_results["Teacher"][0], "Teacher Parallel Analysis")


# 14. Results registry

`results.json` is updated only with **computed results**. No result should be manually invented.

Each table/figure record should contain:

- identifier
- title
- analytical purpose
- data source
- statistical method
- actual result
- interpretation
- manuscript section

This makes the analysis reproducible and keeps the Results chapter synchronized with the code.


In [ ]:
# 14A. RESULTS.JSON UPDATE HELPERS
def load_results():
    if RESULTS_PATH.exists():
        with open(RESULTS_PATH, "r", encoding="utf-8") as f:
            return json.load(f)
    return {"tables": [], "figures": []}

def update_table_result(table_id, result, interpretation=None):
    obj = load_results()
    for t in obj.get("tables", []):
        if t["id"] == table_id:
            t["result"] = result
            if interpretation is not None:
                t["interpretation"] = interpretation
            break
    with open(RESULTS_PATH, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def update_figure_result(figure_id, result, interpretation=None):
    obj = load_results()
    for fig in obj.get("figures", []):
        if fig["id"] == figure_id:
            fig["result"] = result
            if interpretation is not None:
                fig["interpretation"] = interpretation
            break
    with open(RESULTS_PATH, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

print("Results registry helpers ready.")


In [ ]:
# 14B. SAVE GENERATED OUTPUTS
print("="*70)
print("SAVING QPEI OUTPUTS")
print("="*70)

if "inventory" in globals():
    inventory.to_csv(TABLE_DIR / "workbook_inventory.csv", index=False)

for name, obj in [
    ("teacher_item_diagnostics", globals().get("teacher_diag")),
    ("student_item_diagnostics", globals().get("student_diag")),
    ("parent_item_diagnostics", globals().get("parent_diag"))
]:
    if isinstance(obj, pd.DataFrame) and not obj.empty:
        obj.to_csv(TABLE_DIR / f"{name}.csv", index=False)

for label, result in globals().get("efa_results", {}).items():
    _, loadings, diagnostics = result
    loadings.to_csv(TABLE_DIR / f"{label.lower()}_efa_loadings.csv")
    diagnostics.to_csv(TABLE_DIR / f"{label.lower()}_efa_communalities.csv")

if not RESULTS_PATH.exists():
    base_results = {
        "project": {
            "title": "Quality Primary Education Index (QPEI)",
            "data_path": str(DATA_PATH),
            "output_dir": str(OUTPUT_DIR)
        },
        "tables": [],
        "figures": []
    }
    with open(RESULTS_PATH, "w", encoding="utf-8") as f:
        json.dump(base_results, f, ensure_ascii=False, indent=2)

print("results.json:", RESULTS_PATH.exists())
print("Output directory:", OUTPUT_DIR)

files = [p for p in OUTPUT_DIR.rglob("*") if p.is_file()]
print(f"Generated files: {len(files)}")
for p in files:
    print(" •", p.relative_to(OUTPUT_DIR))


# 15. Final analysis checklist

Before writing conclusions, verify:

- [ ] 32 schools retained and correctly identified
- [ ] No unresolved school-ID ambiguity remains in analysis datasets
- [ ] Original Bangla wording preserved
- [ ] English translation dictionary completed from the actual instrument
- [ ] Demographics separated from measurement items
- [ ] Item diagnostics completed
- [ ] Teacher EFA completed
- [ ] Student EFA completed
- [ ] Parent EFA completed
- [ ] Observation/environment treated appropriately for their sample structure
- [ ] Reliability interpreted appropriately
- [ ] CFA run only where estimable and substantively justified
- [ ] Aggregation evidence assessed
- [ ] 35-indicator/QPEI framework crosswalk finalized
- [ ] Primary QPEI uses 20/15/15/20/15/15 weights
- [ ] Robustness analyses completed
- [ ] Demographic/explanatory analyses respect analytical level
- [ ] Every reported result is stored in results.json
- [ ] Every manuscript table/figure traces to a reproducible analysis object
